# 6-5 Autograd 디버깅과 안전한 평가 — 심화

직접 작성한 코드와 저장된 실행 결과를 정리했습니다.


In [1]:
# 검증 가능 정답 코드
# eval 호출만 한 forward와 eval+no-grad forward를 분리해 mode 전환과 graph 기록을 서로 다른 축으로 검사합니다.
import torch
import torch.nn as nn

model = nn.Linear(3, 2)
x = torch.ones(4, 3)
before = model.weight.detach().clone()

model.eval()
out_eval_only = model(x)
with torch.no_grad():
    out_safe = model(x)

# 두 경로 뒤 parameter state도 비교해 graph 비활성화 여부와 optimizer update 부재를 함께 확인합니다.
print(f"eval_only_tracks_grad={out_eval_only.requires_grad}")
print(f"safe_eval_tracks_grad={out_safe.requires_grad}")
print(f"parameters_unchanged={torch.equal(before, model.weight)}")

eval_only_tracks_grad=True
safe_eval_tracks_grad=False
parameters_unchanged=True


In [2]:
# 검증 가능 정답 코드
# train step에만 zeroing·backward·step을 두고 validation은 eval과 no-grad에서 관찰만 수행합니다.
import torch
import torch.nn as nn

torch.manual_seed(4)
model = nn.Linear(2, 2)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
x = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
y = torch.tensor([0, 1], dtype=torch.long)

def train_step(model, optimizer, criterion, x, y):
    model.train()
    optimizer.zero_grad()
    loss = criterion(model(x), y)
    loss.backward()
    optimizer.step()
    return loss.item()

def valid_step(model, criterion, x, y):
    model.eval()
    with torch.no_grad():
        logits = model(x)
        loss = criterion(logits, y)
    return loss.item(), logits.requires_grad

train_value = train_step(model, optimizer, criterion, x, y)
before_valid = [p.detach().clone() for p in model.parameters()]
valid_value, tracks_grad = valid_step(model, criterion, x, y)
unchanged = all(torch.equal(a, b) for a, b in zip(before_valid, model.parameters()))

# 반환 loss의 타입, validation graph 추적, 검증 전후 weight 불변을 각기 확인해 경계를 닫습니다.
print(f"train_loss_is_float={isinstance(train_value, float)}")
print(f"valid_tracks_grad={tracks_grad}")
print(f"weights_unchanged_in_valid={unchanged}")

train_loss_is_float=True
valid_tracks_grad=False
weights_unchanged_in_valid=True


In [3]:
# 검증 가능 정답 코드
# A는 학습 logits를 loss 전에 detach하고, B는 loss 경로를 보존한 채 보고용 metric만 분리하도록 대비합니다.
import torch
import torch.nn as nn

model = nn.Linear(2, 2)
criterion = nn.CrossEntropyLoss()
x = torch.tensor([[1.0, -1.0], [0.5, 1.0]])
y = torch.tensor([0, 1], dtype=torch.long)

logits_a = model(x).detach()
loss_a = criterion(logits_a, y)
a_failed = False
try:
    loss_a.backward()
except RuntimeError:
    a_failed = True

model.zero_grad()
logits_b = model(x)
loss_b = criterion(logits_b, y)
metric_pred = logits_b.detach().argmax(dim=1)
loss_b.backward()

# backward 성공과 weight gradient 존재, metric graph 부재를 함께 검사해 detach 경계를 지킨 B를 승인합니다.
print(f"branch_A_backward_failed={a_failed}")
print(f"branch_B_has_weight_grad={model.weight.grad is not None}")
print(f"metric_tracks_grad={metric_pred.requires_grad}")
print("approved=B")

branch_A_backward_failed=True
branch_B_has_weight_grad=True
metric_tracks_grad=False
approved=B
